<a href="https://colab.research.google.com/github/ebtihalallien-alt/Genspark-Agentic-New-Curation/blob/main/News_Gathering_Tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install feedparser

In [ ]:
import os
import requests
import feedparser


def require_text(value, field_name):

    if not isinstance(value, str) or value.strip() == "":
        raise ValueError(field_name + " must be a non-empty string.")


def get_newsdata_news(query, api_key, max_results=10):

    require_text(query, "query")

    if not isinstance(max_results, int) or max_results < 1 or max_results > 10:
        raise ValueError("max_results must be between 1 and 10.")

    if not api_key:
        raise ValueError("NewsData API key is required.")

    url = "https://newsdata.io/api/1/latest"

    params = {
        "apikey": api_key,
        "q": query,
        "language": "en"
    }

    response = requests.get(
        url,
        params=params,
        timeout=15
    )

    response.raise_for_status()

    data = response.json()

    results = data.get("results", [])

    articles = []

    for item in results[:max_results]:

        articles.append({
            "source": "NewsData.io",
            "title": item.get("title"),
            "description": item.get("description"),
            "url": item.get("link"),
            "published_at": item.get("pubDate")
        })

    return articles


def get_gnews_news(query, api_key, max_results=10):

    require_text(query, "query")

    if not isinstance(max_results, int) or max_results < 1 or max_results > 10:
        raise ValueError("max_results must be between 1 and 10.")

    if not api_key:
        raise ValueError("GNews API key is required.")

    url = "https://gnews.io/api/v4/search"

    params = {
        "q": query,
        "lang": "en",
        "max": max_results,
        "apikey": api_key
    }

    response = requests.get(
        url,
        params=params,
        timeout=15
    )

    response.raise_for_status()

    data = response.json()

    results = data.get("articles", [])

    articles = []

    for item in results[:max_results]:

        source_info = item.get("source", {})

        articles.append({
            "source": "GNews",
            "title": item.get("title"),
            "description": item.get("description"),
            "url": item.get("url"),
            "published_at": item.get("publishedAt"),
            "publisher": source_info.get("name")
        })

    return articles


def get_google_news_rss(query, max_results=10):

    require_text(query, "query")

    if not isinstance(max_results, int) or max_results < 1 or max_results > 10:
        raise ValueError("max_results must be between 1 and 10.")

    rss_url = "https://news.google.com/rss/search"

    feed = feedparser.parse(
        rss_url,
        query={
            "q": query,
            "hl": "en-US",
            "gl": "US",
            "ceid": "US:en"
        }
    )

    articles = []

    for item in feed.entries[:max_results]:

        articles.append({
            "source": "Google News RSS",
            "title": item.get("title"),
            "description": item.get("summary"),
            "url": item.get("link"),
            "published_at": item.get("published")
        })

    return articles